<a href="https://colab.research.google.com/github/KaptainKris92/HuggingFace_RL_Course/blob/main/notebooks/unit1/LunarLander_v3_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LunarLander PPO experiments

This notebook provides a reusable workflow for training, evaluating, comparing and publishing PPO agents for `LunarLander-v3`.

## Core concepts

The environment gives the agent an eight-value observation describing the lander's position, velocity, angle, angular velocity and leg contact. The agent chooses one of four engine actions.

PPO uses an **actor–critic** architecture:

- The **actor** (`pi`) produces a probability distribution over actions.
- The **critic** (`vf`) estimates the expected future return from the current state.
- PPO uses the critic to estimate whether an action performed better or worse than expected, then updates the actor while clipping excessively large policy changes.

Important parameters:

- `gamma`: how strongly future rewards contribute to the return.
- `gae_lambda`: the bias–variance and temporal-credit-assignment trade-off in advantage estimation. It is not the exploration/exploitation setting.
- `ent_coef`: encourages exploration by preventing the actor from becoming deterministic too early.
- `net_arch`: controls the hidden-layer sizes of the actor and critic.

Models are compared on the same fixed evaluation seeds. By default, a candidate replaces the current Hugging Face model only when its fixed-seed mean reward is higher.

#### Installs and imports

In [4]:
!apt-get update -qq
!apt-get install -y -qq swig cmake ffmpeg xvfb libgl1

%pip install -q \
    "stable-baselines3[extra]==2.9.0" \
    "gymnasium[box2d]==1.3.0" \
    "huggingface-hub<2" \
    pyvirtualdisplay

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package swig4.0.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubuntu1_all.deb ...
Unpacking swig (4.0.2-1ubuntu1) ...
Setting up swig4.0 (4.0.2-1ubuntu1) ...
Setting up swig (4.0.2-1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.2 MB/s eta 0:00:00


In [5]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Iterable, Sequence
import json
import os
import shutil
import textwrap

import gymnasium as gym
import numpy as np
import pandas as pd

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    notebook_login,
)
from huggingface_hub.utils import (
    EntryNotFoundError,
    RepositoryNotFoundError,
)

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

from pyvirtualdisplay import Display
from IPython.display import Video, display


if not os.environ.get("DISPLAY"):
    virtual_display = Display(
        visible=False,
        size=(1400, 900),
    )
    virtual_display.start()

    print(
        "Virtual display started:",
        virtual_display.is_alive(),
    )
else:
    print("Using existing display:", os.environ["DISPLAY"])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Virtual display started: True


## Flip model

The plan for this model is to have the same architecture as before, but to change the environment's reward function to reward doing a full 360 degree flip before landing safely.

### Reward wrapper

In [12]:
%%writefile /content/flip_landing_reward_wrapper_v2.py

from __future__ import annotations

import math

import gymnasium as gym
import numpy as np


DEFAULT_REWARD_CONFIG = {
    "flip_landing_bonus": 750.0,
    "rotation_progress_bonus": 150.0,
    "flip_completion_bonus": 200.0,
    "recovery_bonus": 100.0,
    "failed_landing_penalty": 150.0,
    "required_rotations": 1.0,
    "upright_tolerance_radians": 0.30,
    "recovery_angular_velocity_tolerance": 0.50,
    "pre_flip_original_reward_weight": 0.25,
    "post_flip_original_reward_weight": 1.0,
    "post_flip_shaping_weight": 1.0,
    "post_flip_shaping_gamma": 0.995,
    "post_flip_shaping_clip": 10.0,
    "post_flip_center_weight": 50.0,
    "post_flip_horizontal_speed_weight": 30.0,
    "post_flip_angle_weight": 20.0,
    "post_flip_angular_speed_weight": 10.0,
    "post_flip_leg_contact_weight": 10.0,
    "rotation_direction": 1,
    "landing_without_flip_penalty": 300.0,
    "no_flip_terminal_penalty": 300.0,
    "post_flip_vertical_speed_weight": 30.0,
}


class FlipLandingRewardWrapper(gym.Wrapper):
    """
    Stage-aware LunarLander objective:

    1. Complete one full rotation in the selected direction.
    2. Arrest the spin and recover to an upright attitude.
    3. Re-centre, stabilise and land safely.

    The original eight-value observation is extended with:
    - signed rotation progress in [-1, 1];
    - a flip-completed flag;
    - a post-flip-recovery-completed flag.

    Post-flip recovery uses a potential-difference reward based on
    horizontal position, horizontal speed, attitude, angular speed,
    and leg contact. This rewards improvement rather than repeatedly
    paying the agent simply for occupying a favourable state.
    """

    def __init__(
        self,
        env: gym.Env,
        *,
        flip_landing_bonus: float = 750.0,
        rotation_progress_bonus: float = 150.0,
        flip_completion_bonus: float = 200.0,
        recovery_bonus: float = 100.0,
        no_flip_terminal_penalty: float = 0.0,
        failed_landing_penalty: float = 150.0,
        required_rotations: float = 1.0,
        upright_tolerance_radians: float = 0.30,
        recovery_angular_velocity_tolerance: float = 0.50,
        pre_flip_original_reward_weight: float = 0.25,
        post_flip_original_reward_weight: float = 1.0,
        post_flip_shaping_weight: float = 1.0,
        post_flip_shaping_gamma: float = 0.995,
        post_flip_shaping_clip: float = 10.0,
        post_flip_center_weight: float = 50.0,
        post_flip_horizontal_speed_weight: float = 30.0,
        post_flip_angle_weight: float = 20.0,
        post_flip_angular_speed_weight: float = 10.0,
        post_flip_leg_contact_weight: float = 10.0,
        rotation_direction: int = 1,
        landing_without_flip_penalty: float = 300.0,
        post_flip_vertical_speed_weight: float = 30.0,
    ):
        super().__init__(env)

        if required_rotations <= 0:
            raise ValueError("required_rotations must be positive.")

        if rotation_direction not in (-1, 1):
            raise ValueError(
                "rotation_direction must be either -1 or 1."
            )

        if not 0.0 <= post_flip_shaping_gamma <= 1.0:
            raise ValueError(
                "post_flip_shaping_gamma must be in [0, 1]."
            )

        if post_flip_shaping_clip <= 0:
            raise ValueError(
                "post_flip_shaping_clip must be positive."
            )

        self.flip_landing_bonus = float(flip_landing_bonus)
        self.rotation_progress_bonus = float(
            rotation_progress_bonus
        )
        self.flip_completion_bonus = float(
            flip_completion_bonus
        )
        self.recovery_bonus = float(recovery_bonus)
        self.no_flip_terminal_penalty = float(
            no_flip_terminal_penalty
        )
        self.failed_landing_penalty = float(
            failed_landing_penalty
        )

        self.required_rotation = (
            2.0 * math.pi * float(required_rotations)
        )
        self.required_rotations = float(required_rotations)
        self.upright_tolerance_radians = float(
            upright_tolerance_radians
        )
        self.recovery_angular_velocity_tolerance = float(
            recovery_angular_velocity_tolerance
        )

        self.pre_flip_original_reward_weight = float(
            pre_flip_original_reward_weight
        )
        self.post_flip_original_reward_weight = float(
            post_flip_original_reward_weight
        )

        self.post_flip_shaping_weight = float(
            post_flip_shaping_weight
        )
        self.post_flip_shaping_gamma = float(
            post_flip_shaping_gamma
        )
        self.post_flip_shaping_clip = float(
            post_flip_shaping_clip
        )
        self.post_flip_center_weight = float(
            post_flip_center_weight
        )
        self.post_flip_horizontal_speed_weight = float(
            post_flip_horizontal_speed_weight
        )
        self.post_flip_angle_weight = float(
            post_flip_angle_weight
        )
        self.post_flip_angular_speed_weight = float(
            post_flip_angular_speed_weight
        )
        self.post_flip_leg_contact_weight = float(
            post_flip_leg_contact_weight
        )

        self.post_flip_vertical_speed_weight = float(
            post_flip_vertical_speed_weight
        )

        self.rotation_direction = int(rotation_direction)

        self.previous_angle = 0.0
        self.cumulative_rotation = 0.0
        self.maximum_rotation_progress = 0.0
        self.flip_completed = False
        self.recovery_completed = False
        self.previous_post_flip_potential: float | None = None
        self.landing_without_flip_penalty = float(
            landing_without_flip_penalty
        )

        base_space = env.observation_space

        if not isinstance(base_space, gym.spaces.Box):
            raise TypeError(
                "The wrapped environment must have a Box "
                "observation space."
            )

        extra_low = np.asarray(
            [-1.0, 0.0, 0.0],
            dtype=np.float32,
        )
        extra_high = np.asarray(
            [1.0, 1.0, 1.0],
            dtype=np.float32,
        )

        self.observation_space = gym.spaces.Box(
            low=np.concatenate(
                [
                    base_space.low.astype(np.float32),
                    extra_low,
                ]
            ),
            high=np.concatenate(
                [
                    base_space.high.astype(np.float32),
                    extra_high,
                ]
            ),
            dtype=np.float32,
        )

    @staticmethod
    def _wrap_angle(angle: float) -> float:
        """Wrap an angle to [-pi, pi]."""

        return float(
            np.arctan2(
                np.sin(angle),
                np.cos(angle),
            )
        )

    def _signed_rotation_progress(self) -> float:
        return float(
            np.clip(
                self.cumulative_rotation
                / self.required_rotation,
                -1.0,
                1.0,
            )
        )

    def _augment_observation(
        self,
        observation: np.ndarray,
    ) -> np.ndarray:
        return np.concatenate(
            [
                np.asarray(
                    observation,
                    dtype=np.float32,
                ),
                np.asarray(
                    [
                        self._signed_rotation_progress(),
                        float(self.flip_completed),
                        float(self.recovery_completed),
                    ],
                    dtype=np.float32,
                ),
            ]
        )

    def _post_flip_potential(
        self,
        observation: np.ndarray,
    ) -> float:
        """
        Higher values represent safer post-flip states.

        LunarLander-v3 observation positions:
        0 x position, 2 x velocity, 4 angle, 5 angular velocity,
        6 left-leg contact and 7 right-leg contact.
        """

        x_position = float(
            observation[0]
        )
        horizontal_speed = float(
            observation[2]
        )
        vertical_speed = float(
            observation[3]
        )

        angle = abs(
            self._wrap_angle(
                float(observation[4])
            )
        )

        angular_speed = abs(
            float(observation[5])
        )

        leg_contacts = (
            float(observation[6] > 0.5)
            + float(observation[7] > 0.5)
        )

        return float(
            -self.post_flip_center_weight
            * abs(x_position)

            -self.post_flip_horizontal_speed_weight
            * abs(horizontal_speed)

            -self.post_flip_vertical_speed_weight
            * abs(vertical_speed)

            -self.post_flip_angle_weight
            * angle

            -self.post_flip_angular_speed_weight
            * angular_speed

            +self.post_flip_leg_contact_weight
            * leg_contacts
        )

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict | None = None,
    ):
        observation, info = self.env.reset(
            seed=seed,
            options=options,
        )

        self.previous_angle = float(observation[4])
        self.cumulative_rotation = 0.0
        self.maximum_rotation_progress = 0.0
        self.flip_completed = False
        self.recovery_completed = False
        self.previous_post_flip_potential = None

        info = dict(info)
        info.update(
            {
                "flip_completed": False,
                "recovery_completed": False,
                "rotation_progress": 0.0,
                "rotations_completed": 0.0,
                "rotation_progress_reward": 0.0,
                "flip_completion_reward": 0.0,
                "recovery_reward": 0.0,
                "post_flip_shaping_reward": 0.0,
                "flip_landing_bonus": 0.0,
                "terminal_adjustment": 0.0,
            }
        )

        return self._augment_observation(observation), info

    def step(self, action):
        (
            observation,
            original_reward,
            terminated,
            truncated,
            info,
        ) = self.env.step(action)

        current_angle = float(observation[4])
        angular_velocity = float(observation[5])

        angle_change = self._wrap_angle(
            current_angle - self.previous_angle
        )
        self.previous_angle = current_angle

        directed_change = (
            self.rotation_direction * angle_change
        )
        self.cumulative_rotation += directed_change

        rotation_progress = float(
            np.clip(
                self.cumulative_rotation
                / self.required_rotation,
                0.0,
                1.0,
            )
        )

        new_progress = max(
            0.0,
            rotation_progress
            - self.maximum_rotation_progress,
        )
        progress_reward = (
            self.rotation_progress_bonus
            * new_progress
        )
        self.maximum_rotation_progress = max(
            self.maximum_rotation_progress,
            rotation_progress,
        )

        completion_reward = 0.0

        if (
            rotation_progress >= 1.0
            and not self.flip_completed
        ):
            self.flip_completed = True
            completion_reward = (
                self.flip_completion_bonus
            )

        wrapped_angle = abs(
            self._wrap_angle(current_angle)
        )
        upright = (
            wrapped_angle
            <= self.upright_tolerance_radians
        )

        recovery_reward = 0.0

        if (
            self.flip_completed
            and not self.recovery_completed
            and upright
            and abs(angular_velocity)
            <= self.recovery_angular_velocity_tolerance
        ):
            self.recovery_completed = True
            recovery_reward = self.recovery_bonus

        post_flip_shaping_reward = 0.0
        post_flip_potential = None

        if self.flip_completed:
            post_flip_potential = (
                self._post_flip_potential(observation)
            )

            if self.previous_post_flip_potential is None:
                self.previous_post_flip_potential = (
                    post_flip_potential
                )
            else:
                potential_difference = (
                    self.post_flip_shaping_gamma
                    * post_flip_potential
                    - self.previous_post_flip_potential
                )
                post_flip_shaping_reward = float(
                    np.clip(
                        self.post_flip_shaping_weight
                        * potential_difference,
                        -self.post_flip_shaping_clip,
                        self.post_flip_shaping_clip,
                    )
                )
                self.previous_post_flip_potential = (
                    post_flip_potential
                )

        left_leg_contact = bool(observation[6] > 0.5)
        right_leg_contact = bool(observation[7] > 0.5)

        landed_safely = bool(
            terminated
            and float(original_reward) > 0
            and left_leg_contact
            and right_leg_contact
            and upright
        )

        episode_finished = bool(
            terminated or truncated
        )

        terminal_adjustment = 0.0
        outcome = "in_progress"

        if self.flip_completed and landed_safely:
            terminal_adjustment = (
                self.flip_landing_bonus
            )
            outcome = "flip_and_safe_landing"

        elif episode_finished:
            if not self.flip_completed:
                if landed_safely:
                    terminal_adjustment = -(
                        self.landing_without_flip_penalty
                    )
                    outcome = (
                        "safe_landing_without_flip"
                    )
                else:
                    terminal_adjustment = -(
                        self.no_flip_terminal_penalty
                    )
                    outcome = (
                        "episode_ended_without_flip"
                    )

            else:
                terminal_adjustment = -(
                    self.failed_landing_penalty
                )
                outcome = (
                    "flip_but_failed_landing"
                )

        original_weight = (
            self.post_flip_original_reward_weight
            if self.flip_completed
            else self.pre_flip_original_reward_weight
        )

        modified_reward = (
            original_weight * float(original_reward)
            + progress_reward
            + completion_reward
            + recovery_reward
            + post_flip_shaping_reward
            + terminal_adjustment
        )

        info = dict(info)
        info.update(
            {
                "original_reward": float(original_reward),
                "rotation_progress": rotation_progress,
                "rotation_progress_reward": progress_reward,
                "flip_completion_reward": completion_reward,
                "recovery_reward": recovery_reward,
                "post_flip_shaping_reward": (
                    post_flip_shaping_reward
                ),
                "post_flip_potential": post_flip_potential,
                "cumulative_rotation": (
                    self.cumulative_rotation
                ),
                "rotations_completed": (
                    self.cumulative_rotation
                    / (2.0 * math.pi)
                ),
                "flip_completed": self.flip_completed,
                "recovery_completed": (
                    self.recovery_completed
                ),
                "landed_safely": landed_safely,
                "original_reward_weight": original_weight,
                "terminal_adjustment": (
                    terminal_adjustment
                ),
                "flip_landing_bonus": (
                    self.flip_landing_bonus
                    if outcome
                    == "flip_and_safe_landing"
                    else 0.0
                ),
                "custom_outcome": outcome,
            }
        )

        return (
            self._augment_observation(observation),
            modified_reward,
            terminated,
            truncated,
            info,
        )


def make_flip_lander(
    *,
    render_mode: str | None = None,
    reward_config: dict | None = None,
) -> gym.Env:
    """
    Build the exact environment used for training and evaluation.

    Pass the same reward_config to the uploader so the Hub model card
    records the environment accurately.
    """

    config = {
        **DEFAULT_REWARD_CONFIG,
        **(reward_config or {}),
    }

    base_env = gym.make(
        "LunarLander-v3",
        render_mode=render_mode,
    )

    return FlipLandingRewardWrapper(
        base_env,
        **config,
    )


Overwriting /content/flip_landing_reward_wrapper_v2.py


In [13]:
from copy import deepcopy
from pathlib import Path
import importlib

import flip_landing_reward_wrapper_v2 as flip_wrapper

# Important when the module has previously been imported in this runtime.
importlib.reload(flip_wrapper)

DEFAULT_REWARD_CONFIG = (
    flip_wrapper.DEFAULT_REWARD_CONFIG
)
FlipLandingRewardWrapper = (
    flip_wrapper.FlipLandingRewardWrapper
)
make_flip_lander = (
    flip_wrapper.make_flip_lander
)

REWARD_WRAPPER_PATH = Path(
    "/content/flip_landing_reward_wrapper_v2.py"
)

base_reward_config = deepcopy(
    DEFAULT_REWARD_CONFIG
)

base_reward_config[
    "post_flip_shaping_gamma"
] = 0.999


flip_acquisition_config = {
    **base_reward_config,

    # Make discovering the flip worthwhile.
    "rotation_progress_bonus": 300.0,
    "flip_completion_bonus": 500.0,

    # Retain some position and velocity guidance,
    # but prevent ordinary landing from winning.
    "pre_flip_original_reward_weight": 0.15,
    "post_flip_original_reward_weight": 1.0,

    "landing_without_flip_penalty": 300.0,
    "no_flip_terminal_penalty": 100.0,

    # During acquisition, a flip followed by a crash
    # must still be recognised as progress.
    "failed_landing_penalty": 0.0,

    "recovery_bonus": 100.0,
    "post_flip_shaping_weight": 0.5,

    "flip_landing_bonus": 1_000.0,
}


flip_landing_config = {
    **flip_acquisition_config,

    # Keep the flip rewards. Reducing them risks
    # catastrophic forgetting of the rotation.
    "rotation_progress_bonus": 300.0,
    "flip_completion_bonus": 500.0,

    # The requested post-flip landing multiplier.
    "post_flip_original_reward_weight": 3.0,

    "recovery_bonus": 250.0,
    "post_flip_shaping_weight": 1.5,

    # Once the flip is learned, crashes should
    # become significantly unattractive.
    "failed_landing_penalty": 400.0,

    # Exclusive reward for completing both stages.
    "flip_landing_bonus": 1_500.0,
}


def make_flip_acquisition_lander(
    *,
    render_mode: str | None = None,
):
    return make_flip_lander(
        render_mode=render_mode,
        reward_config=(
            flip_acquisition_config
        ),
    )


def make_flip_landing_lander(
    *,
    render_mode: str | None = None,
):
    return make_flip_lander(
        render_mode=render_mode,
        reward_config=(
            flip_landing_config
        ),
    )

### Upload checkpoint to HuggingFace

In [14]:
from pathlib import Path

from huggingface_hub import HfApi
from stable_baselines3.common.callbacks import (
    BaseCallback,
)


class HubCheckpointCallback(
    BaseCallback
):
    def __init__(
        self,
        *,
        repo_id: str,
        every_timesteps: int = 1_000_000,
        verbose: int = 1,
    ):
        super().__init__(verbose)

        self.repo_id = repo_id
        self.every_timesteps = int(
            every_timesteps
        )
        self.next_upload = int(
            every_timesteps
        )
        self.api = HfApi()

        self.local_path = Path(
            "/content/"
            "latest_flip_checkpoint.zip"
        )

    def _on_step(self) -> bool:
        if (
            self.num_timesteps
            < self.next_upload
        ):
            return True

        step = int(
            self.num_timesteps
        )

        self.model.save(
            str(self.local_path)
        )

        self.api.upload_file(
            path_or_fileobj=str(
                self.local_path
            ),
            path_in_repo=(
                "training-checkpoints/"
                f"ppo_flip_{step}_steps.zip"
            ),
            repo_id=self.repo_id,
            repo_type="model",
            commit_message=(
                f"Save training checkpoint "
                f"at {step:,} steps"
            ),
        )

        if self.verbose:
            print(
                "Uploaded Hub checkpoint:",
                f"{step:,} steps",
            )

        while (
            self.next_upload
            <= step
        ):
            self.next_upload += (
                self.every_timesteps
            )

        return True

### Model loading ane evaluation

In [ ]:
def load_ppo_model(
    model_or_path: PPO | str | Path,
    *,
    device: str = "cpu",
) -> PPO:
    """Return an existing PPO object or load one from disk."""

    if isinstance(model_or_path, PPO):
        return model_or_path

    model_path = Path(model_or_path)

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}"
        )

    return PPO.load(
        str(model_path),
        device=device,
    )


def normalise_action(action: Any) -> Any:
    """
    Convert a one-element discrete-action array into an integer.

    PPO commonly returns array([2]), while a non-vectorised
    LunarLander environment expects the integer 2.
    """

    action_array = np.asarray(action)

    if action_array.size == 1:
        return int(action_array.item())

    return action


def evaluate_model(
    model_or_path: PPO | str | Path,
    *,
    seeds: Iterable[int],
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    deterministic: bool = True,
    device: str = "cpu",
) -> dict[str, Any]:
    """
    Evaluate a model on an explicit set of episode seeds.

    Explicit seeds make comparisons between models reproducible
    and ensure they face the same landing scenarios.
    """

    model = load_ppo_model(
        model_or_path,
        device=device,
    )

    env_kwargs = dict(env_kwargs or {})
    seed_list = [int(seed) for seed in seeds]

    if not seed_list:
        raise ValueError(
            "At least one evaluation seed is required."
        )

    episode_rewards: list[float] = []
    episode_lengths: list[int] = []

    for seed in seed_list:
        env = Monitor(
            gym.make(
                env_id,
                **env_kwargs,
            )
        )

        try:
            observation, info = env.reset(seed=seed)

            terminated = False
            truncated = False
            total_reward = 0.0
            episode_length = 0

            while not (terminated or truncated):
                action, _ = model.predict(
                    observation,
                    deterministic=deterministic,
                )

                (
                    observation,
                    reward,
                    terminated,
                    truncated,
                    info,
                ) = env.step(
                    normalise_action(action)
                )

                total_reward += float(reward)
                episode_length += 1

            episode_rewards.append(total_reward)
            episode_lengths.append(episode_length)

        finally:
            env.close()

    rewards = np.asarray(
        episode_rewards,
        dtype=float,
    )
    lengths = np.asarray(
        episode_lengths,
        dtype=float,
    )

    mean_reward = float(rewards.mean())
    std_reward = float(rewards.std())

    return {
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "course_score": mean_reward - std_reward,
        "success_rate_200": float(
            np.mean(rewards >= 200.0)
        ),
        "mean_episode_length": float(
            lengths.mean()
        ),
        "min_reward": float(rewards.min()),
        "max_reward": float(rewards.max()),
        "n_episodes": len(seed_list),
        "rewards": rewards.tolist(),
        "episode_lengths": lengths.astype(int).tolist(),
    }


def metrics_only(
    results: dict[str, Any],
) -> dict[str, float | int]:
    """Return only summary metrics, excluding episode arrays."""

    return {
        "mean_reward": results["mean_reward"],
        "std_reward": results["std_reward"],
        "course_score": results["course_score"],
        "success_rate_200": results["success_rate_200"],
        "mean_episode_length": (
            results["mean_episode_length"]
        ),
        "min_reward": results["min_reward"],
        "max_reward": results["max_reward"],
        "n_episodes": results["n_episodes"],
    }


def compare_candidate_to_hub(
    candidate_model_or_path: PPO | str | Path,
    *,
    repo_id: str,
    hub_model_filename: str,
    evaluation_seeds: Iterable[int],
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    selection_metric: str = "mean_reward",
    min_improvement: float = 0.0,
    device: str = "cpu",
) -> dict[str, Any]:
    """
    Compare a candidate with the current Hub model on the same seeds.

    Valid selection metrics:
    - mean_reward
    - course_score
    - success_rate_200
    """

    valid_metrics = {
        "mean_reward",
        "course_score",
        "success_rate_200",
    }

    if selection_metric not in valid_metrics:
        raise ValueError(
            "selection_metric must be one of "
            f"{sorted(valid_metrics)}."
        )

    seed_list = [
        int(seed)
        for seed in evaluation_seeds
    ]

    candidate_results = evaluate_model(
        candidate_model_or_path,
        seeds=seed_list,
        env_id=env_id,
        env_kwargs=env_kwargs,
        device=device,
    )

    current_results = None
    current_model_path = None

    try:
        current_model_path = hf_hub_download(
            repo_id=repo_id,
            filename=hub_model_filename,
            repo_type="model",
        )

        current_results = evaluate_model(
            current_model_path,
            seeds=seed_list,
            env_id=env_id,
            env_kwargs=env_kwargs,
            device=device,
        )

    except (
        EntryNotFoundError,
        RepositoryNotFoundError,
    ):
        print(
            "No current Hub model was found. "
            "The candidate is eligible for upload."
        )

    rows = [
        {
            "model": "candidate",
            **metrics_only(candidate_results),
        }
    ]

    if current_results is not None:
        rows.append(
            {
                "model": "current_hub",
                **metrics_only(current_results),
            }
        )

    comparison_table = (
        pd.DataFrame(rows)
        .set_index("model")
    )

    if current_results is None:
        improvement = None
        candidate_is_better = True

    else:
        candidate_value = float(
            candidate_results[selection_metric]
        )
        current_value = float(
            current_results[selection_metric]
        )

        improvement = (
            candidate_value - current_value
        )

        candidate_is_better = (
            candidate_value
            > current_value + min_improvement
        )

    display(comparison_table.round(3))

    print("Selection metric:", selection_metric)

    if improvement is not None:
        print(f"Improvement: {improvement:+.3f}")

    print(
        "Candidate is eligible for upload:",
        candidate_is_better,
    )

    return {
        "candidate": candidate_results,
        "current": current_results,
        "current_model_path": current_model_path,
        "selection_metric": selection_metric,
        "min_improvement": min_improvement,
        "improvement": improvement,
        "candidate_is_better": candidate_is_better,
        "table": comparison_table,
    }

### Training function

In [15]:
# New training function
from collections.abc import Callable, Sequence
from pathlib import Path
from typing import Any
import json

import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor


def train_ppo_experiment(
    *,
    experiment_name: str,
    total_timesteps: int,
    env_id: str = "LunarLander-v3",
    env_factory: Callable[[], gym.Env] | None = None,
    env_kwargs: dict[str, Any] | None = None,
    n_envs: int = 16,
    seed: int = 42,
    actor_layers: Sequence[int] = (64, 64),
    critic_layers: Sequence[int] = (64, 64),
    learning_rate: float = 3e-4,
    n_steps: int = 1024,
    batch_size: int = 64,
    n_epochs: int = 4,
    gamma: float = 0.999,
    gae_lambda: float = 0.98,
    ent_coef: float = 0.01,
    clip_range: float = 0.2,
    eval_every_timesteps: int = 50_000,
    checkpoint_every_timesteps: int = 100_000,
    n_eval_episodes: int = 20,
    output_root: str | Path = "/content/rl-experiments",
    device: str = "cpu",
    verbose: int = 1,
    extra_callbacks: Sequence[
        BaseCallback
    ] | None = None,
    initial_model_path: (
        str | Path | None
    ) = None,
) -> dict[str, Any]:
    """
    Train a PPO actor–critic model.

    env_factory
        Optional callable that returns a custom Gymnasium environment.
        Use this for the flip-reward environment.

    The function saves:
    - periodic safety checkpoints;
    - the checkpoint with the highest evaluation mean;
    - the model from the final training timestep;
    - the complete training configuration.
    """

    if not experiment_name.strip():
        raise ValueError(
            "experiment_name must not be empty."
        )

    if total_timesteps <= 0:
        raise ValueError(
            "total_timesteps must be positive."
        )

    if n_envs <= 0:
        raise ValueError(
            "n_envs must be positive."
        )

    env_kwargs = dict(env_kwargs or {})

    if env_factory is not None and env_kwargs:
        raise ValueError(
            "Use either env_factory or env_kwargs, not both. "
            "Put custom environment arguments inside the factory."
        )

    output_dir = (
        Path(output_root)
        / experiment_name
    )
    best_model_dir = (
        output_dir
        / "best_model"
    )
    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )
    evaluation_dir = (
        output_dir
        / "evaluations"
    )

    for directory in (
        best_model_dir,
        checkpoint_dir,
        evaluation_dir,
    ):
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    configuration = {
        "experiment_name": experiment_name,
        "env_id": env_id,
        "environment_factory": (
            getattr(
                env_factory,
                "__name__",
                None,
            )
            if env_factory is not None
            else None
        ),
        "env_kwargs": env_kwargs,
        "total_timesteps": total_timesteps,
        "n_envs": n_envs,
        "seed": seed,
        "actor_layers": list(actor_layers),
        "critic_layers": list(critic_layers),
        "learning_rate": learning_rate,
        "n_steps": n_steps,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "ent_coef": ent_coef,
        "clip_range": clip_range,
        "eval_every_timesteps": (
            eval_every_timesteps
        ),
        "checkpoint_every_timesteps": (
            checkpoint_every_timesteps
        ),
        "n_eval_episodes": n_eval_episodes,
        "device": device,
        "initial_model_path": (
            str(initial_model_path)
            if initial_model_path is not None
            else None
        ),
    }

    config_path = (
        output_dir
        / "training_config.json"
    )

    config_path.write_text(
        json.dumps(
            configuration,
            indent=2,
        ),
        encoding="utf-8",
    )

    # Create standard or customised environments.
    if env_factory is None:
        train_env = make_vec_env(
            env_id,
            n_envs=n_envs,
            seed=seed,
            env_kwargs=env_kwargs,
        )

        eval_env = Monitor(
            gym.make(
                env_id,
                **env_kwargs,
            )
        )

    else:
        train_env = make_vec_env(
            env_factory,
            n_envs=n_envs,
            seed=seed,
        )

        eval_env = Monitor(
            env_factory()
        )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(
            best_model_dir
        ),
        log_path=str(
            evaluation_dir
        ),
        eval_freq=max(
            eval_every_timesteps // n_envs,
            1,
        ),
        n_eval_episodes=n_eval_episodes,
        deterministic=True,
        render=False,
        verbose=1,
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=max(
            checkpoint_every_timesteps // n_envs,
            1,
        ),
        save_path=str(
            checkpoint_dir
        ),
        name_prefix=experiment_name,
        verbose=2,
    )

    callbacks = [
        eval_callback,
        checkpoint_callback,
        *(extra_callbacks or []),
    ]

    policy_kwargs = {
        "net_arch": {
            "pi": list(actor_layers),
            "vf": list(critic_layers),
        }
    }

    if initial_model_path is None:
        model = PPO(
            policy="MlpPolicy",
            env=train_env,
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            n_epochs=n_epochs,
            gamma=gamma,
            gae_lambda=gae_lambda,
            ent_coef=ent_coef,
            clip_range=clip_range,
            policy_kwargs=policy_kwargs,
            device=device,
            seed=seed,
            verbose=verbose,
        )

        reset_num_timesteps = True

    else:
        model = PPO.load(
            str(initial_model_path),
            env=train_env,
            device=device,
        )

        reset_num_timesteps = False

    try:
        model.learn(
            total_timesteps=total_timesteps,
            callback=callbacks,
            progress_bar=True,
            reset_num_timesteps=(
                reset_num_timesteps
            ),
        )

        final_model_stem = (
            output_dir
            / "final_model"
        )

        model.save(
            str(final_model_stem)
        )

    finally:
        train_env.close()
        eval_env.close()

    best_model_path = (
        best_model_dir
        / "best_model.zip"
    )
    final_model_path = (
        output_dir
        / "final_model.zip"
    )
    evaluations_path = (
        evaluation_dir
        / "evaluations.npz"
    )

    if not best_model_path.exists():
        raise FileNotFoundError(
            "EvalCallback did not create best_model.zip. "
            "Check that training reached at least one "
            "evaluation interval."
        )

    print("Best model:", best_model_path)
    print("Final model:", final_model_path)
    print("Configuration:", config_path)

    return {
        "experiment_name": experiment_name,
        "output_dir": output_dir,
        "best_model_path": best_model_path,
        "final_model_path": final_model_path,
        "evaluations_path": evaluations_path,
        "config_path": config_path,
        "configuration": configuration,
    }

### Evaluation function

In [16]:
import numpy as np
import pandas as pd


def evaluate_flip_model(
    model_or_path,
    *,
    reward_config: dict,
    n_episodes: int = 100,
    starting_seed: int = 20_000,
    deterministic: bool = True,
):
    """
    Evaluate the flip-recover-land agent over fixed seeds.
    """

    if n_episodes <= 0:
        raise ValueError(
            "n_episodes must be positive."
        )

    model = load_ppo_model(
        model_or_path
    )

    episode_rows = []

    for episode_number in range(
        n_episodes
    ):
        seed = (
            starting_seed
            + episode_number
        )

        env = make_flip_lander(
            reward_config=reward_config
        )

        try:
            observation, _ = env.reset(
                seed=seed
            )

            terminated = False
            truncated = False

            shaped_reward_total = 0.0
            original_reward_total = 0.0
            episode_steps = 0
            final_info = {}

            while not (
                terminated or truncated
            ):
                action, _ = model.predict(
                    observation,
                    deterministic=deterministic,
                )

                # LunarLander-v3 uses Discrete(4).
                action = int(
                    np.asarray(
                        action
                    ).item()
                )

                (
                    observation,
                    shaped_reward,
                    terminated,
                    truncated,
                    info,
                ) = env.step(action)

                shaped_reward_total += float(
                    shaped_reward
                )

                original_reward_total += float(
                    info.get(
                        "original_reward",
                        shaped_reward,
                    )
                )

                episode_steps += 1
                final_info = dict(info)

            flip_completed = bool(
                final_info.get(
                    "flip_completed",
                    False,
                )
            )

            recovery_completed = bool(
                final_info.get(
                    "recovery_completed",
                    False,
                )
            )

            landed_safely = bool(
                final_info.get(
                    "landed_safely",
                    False,
                )
            )

            custom_outcome = str(
                final_info.get(
                    "custom_outcome",
                    "unknown",
                )
            )

            flip_and_landed = (
                custom_outcome
                == "flip_and_safe_landing"
            )

            episode_rows.append(
                {
                    "seed": seed,
                    "shaped_reward": (
                        shaped_reward_total
                    ),
                    "original_reward": (
                        original_reward_total
                    ),
                    "steps": episode_steps,
                    "flip_completed": (
                        flip_completed
                    ),
                    "recovery_completed": (
                        recovery_completed
                    ),
                    "landed_safely": (
                        landed_safely
                    ),
                    "flip_and_landed": (
                        flip_and_landed
                    ),
                    "flip_bonus": float(
                        final_info.get(
                            "flip_landing_bonus",
                            0.0,
                        )
                    ),
                    "rotations_completed": float(
                        final_info.get(
                            "rotations_completed",
                            0.0,
                        )
                    ),
                    "terminal_adjustment": float(
                        final_info.get(
                            "terminal_adjustment",
                            0.0,
                        )
                    ),
                    "custom_outcome": (
                        custom_outcome
                    ),
                }
            )

        finally:
            env.close()

    episodes = pd.DataFrame(
        episode_rows
    )

    shaped_rewards = episodes[
        "shaped_reward"
    ].to_numpy(dtype=float)

    original_rewards = episodes[
        "original_reward"
    ].to_numpy(dtype=float)

    flip_mask = episodes[
        "flip_completed"
    ]

    if flip_mask.any():
        recovery_given_flip_rate = float(
            episodes.loc[
                flip_mask,
                "recovery_completed",
            ].mean()
        )

        landing_given_flip_rate = float(
            episodes.loc[
                flip_mask,
                "flip_and_landed",
            ].mean()
        )
    else:
        recovery_given_flip_rate = 0.0
        landing_given_flip_rate = 0.0

    summary = {
        "n_episodes": int(n_episodes),

        "mean_shaped_reward": float(
            shaped_rewards.mean()
        ),
        "std_shaped_reward": float(
            shaped_rewards.std()
        ),
        "shaped_course_score": float(
            shaped_rewards.mean()
            - shaped_rewards.std()
        ),
        "min_shaped_reward": float(
            shaped_rewards.min()
        ),
        "max_shaped_reward": float(
            shaped_rewards.max()
        ),

        "mean_original_reward": float(
            original_rewards.mean()
        ),
        "std_original_reward": float(
            original_rewards.std()
        ),
        "original_course_score": float(
            original_rewards.mean()
            - original_rewards.std()
        ),
        "min_original_reward": float(
            original_rewards.min()
        ),
        "max_original_reward": float(
            original_rewards.max()
        ),

        "original_reward_200_rate": float(
            np.mean(
                original_rewards >= 200.0
            )
        ),
        "flip_completion_rate": float(
            episodes[
                "flip_completed"
            ].mean()
        ),
        "recovery_completion_rate": float(
            episodes[
                "recovery_completed"
            ].mean()
        ),
        "recovery_given_flip_rate": (
            recovery_given_flip_rate
        ),
        "safe_landing_rate": float(
            episodes[
                "landed_safely"
            ].mean()
        ),
        "flip_landing_rate": float(
            episodes[
                "flip_and_landed"
            ].mean()
        ),
        "landing_given_flip_rate": (
            landing_given_flip_rate
        ),
    }

    print("Flip model evaluation")
    print("---------------------")
    print(
        "Mean shaped reward:",
        f"{summary['mean_shaped_reward']:.2f}",
    )
    print(
        "Mean original reward:",
        f"{summary['mean_original_reward']:.2f}",
    )
    print(
        "Completed a full rotation:",
        f"{summary['flip_completion_rate']:.1%}",
    )
    print(
        "Recovered after the flip:",
        f"{summary['recovery_completion_rate']:.1%}",
    )
    print(
        "Recovery rate given a flip:",
        f"{summary['recovery_given_flip_rate']:.1%}",
    )
    print(
        "Landed safely:",
        f"{summary['safe_landing_rate']:.1%}",
    )
    print(
        "Flipped and landed safely:",
        f"{summary['flip_landing_rate']:.1%}",
    )
    print(
        "Landing rate given a flip:",
        f"{summary['landing_given_flip_rate']:.1%}",
    )

    return {
        "summary": summary,
        "episodes": episodes,
    }

### Train model

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [26]:
# Clean progress display
from rich import get_console

console = get_console()

# Rich versions that store a single active display.
active_live = getattr(console, "_live", None)

if active_live is not None:
    try:
        active_live.stop()
    except Exception:
        try:
            console.clear_live()
        except Exception:
            pass

# Newer Rich versions that store a stack of displays.
live_stack = getattr(console, "_live_stack", None)

if live_stack:
    while live_stack:
        active_live = live_stack[-1]

        try:
            active_live.stop()
        except Exception:
            try:
                console.clear_live()
            except Exception:
                break

print("Rich live display cleared.")

Rich live display cleared.


In [27]:
hub_checkpoint_callback = (
    HubCheckpointCallback(
        repo_id=(
            "KaptainKris92/"
            "ppo-LunarLander-v3-flip"
        ),
        every_timesteps=1_000_000,
    )
)

In [28]:
phase_a_run = train_ppo_experiment(
    experiment_name=(
        "ppo_lunarlander_flip_phase_a"
    ),
    total_timesteps=5_000_000,

    env_factory=(
        make_flip_acquisition_lander
    ),

    n_envs=16,
    seed=42,

    actor_layers=(128, 128),
    critic_layers=(128, 128),

    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.03,
    clip_range=0.2,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=20,

    # extra_callbacks=[
    #     hub_checkpoint_callback
    # ],

    device="cpu",
)

Output()

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.8     |
|    ep_rew_mean     | -110     |
| time/              |          |
|    fps             | 3143     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 95          |
|    ep_rew_mean          | -78.7       |
| time/                   |             |
|    fps                  | 1888        |
|    iterations           | 2           |
|    time_elapsed         | 17          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.011606754 |
|    clip_fraction        | 0.0868      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | 0.00352     |
|    learning

Eval num_timesteps=50000, episode_reward=84.85 +/- 209.83

Episode length: 69.70 +/- 14.11

-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 69.7        |
|    mean_reward          | 84.8        |
| time/                   |             |
|    total_timesteps      | 50000       |
| train/                  |             |
|    approx_kl            | 0.011082446 |
|    clip_fraction        | 0.145       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.31       |
|    explained_variance   | 0.766       |
|    learning_rate        | 0.0003      |
|    loss                 | 51.1        |
|    n_updates            | 12          |
|    policy_gradient_loss | -0.0106     |
|    value_loss           | 156         |
-----------------------------------------


...

Episode length: 126.95 +/- 32.48

-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 127         |
|    mean_reward          | 469         |
| time/                   |             |
|    total_timesteps      | 5000000     |
| train/                  |             |
|    approx_kl            | 0.006599929 |
|    clip_fraction        | 0.0242      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.747      |
|    explained_variance   | 0.847       |
|    learning_rate        | 0.0003      |
|    loss                 | 2.18e+03    |
|    n_updates            | 1220        |
|    policy_gradient_loss | -0.00193    |
|    value_loss           | 5.36e+03    |
-----------------------------------------


Saving model checkpoint to 
/content/rl-experiments/ppo_lunarlander_flip_phase_a/checkpoints/ppo_lunarlander_flip_phase_a_5000000_steps.zip

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 124      |
|    ep_rew_mean     | 425      |
| time/              |          |
|    fps             | 1579     |
|    iterations      | 306      |
|    time_elapsed    | 3174     |
|    total_timesteps | 5013504  |
---------------------------------


Best model: /content/rl-experiments/ppo_lunarlander_flip_phase_a/best_model/best_model.zip
Final model: /content/rl-experiments/ppo_lunarlander_flip_phase_a/final_model.zip
Configuration: /content/rl-experiments/ppo_lunarlander_flip_phase_a/training_config.json


In [29]:
import re

import pandas as pd


def checkpoint_step(
    path: Path,
) -> int:
    match = re.search(
        r"_(\d+)_steps\.zip$",
        path.name,
    )

    return (
        int(match.group(1))
        if match
        else -1
    )


phase_a_checkpoint_paths = sorted(
    (
        phase_a_run["output_dir"]
        / "checkpoints"
    ).glob("*.zip"),
    key=checkpoint_step,
)

phase_a_candidate_paths = [
    Path(
        phase_a_run["best_model_path"]
    ),
    Path(
        phase_a_run["final_model_path"]
    ),
    *phase_a_checkpoint_paths,
]

phase_a_candidate_paths = list(
    dict.fromkeys(
        path
        for path in phase_a_candidate_paths
        if path.exists()
    )
)

phase_a_rows = []

for candidate_path in (
    phase_a_candidate_paths
):
    candidate_evaluation = (
        evaluate_flip_model(
            candidate_path,
            reward_config=(
                flip_acquisition_config
            ),
            n_episodes=20,
            starting_seed=20_000,
        )
    )

    phase_a_rows.append(
        {
            "model_path": str(
                candidate_path
            ),
            **candidate_evaluation[
                "summary"
            ],
        }
    )

phase_a_results = pd.DataFrame(
    phase_a_rows
)

phase_a_results = (
    phase_a_results
    .sort_values(
        [
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "mean_shaped_reward",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    phase_a_results[
        [
            "model_path",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "safe_landing_rate",
            "mean_shaped_reward",
        ]
    ]
)

selected_phase_a_model_path = Path(
    phase_a_results.loc[
        0,
        "model_path",
    ]
)

phase_a_flip_rate = float(
    phase_a_results.loc[
        0,
        "flip_completion_rate",
    ]
)

print(
    "Selected Phase A model:",
    selected_phase_a_model_path,
)

print(
    "Phase A flip rate:",
    f"{phase_a_flip_rate:.1%}",
)

if phase_a_flip_rate == 0.0:
    raise RuntimeError(
        "No Phase A checkpoint completed a flip. "
        "Do not start Phase B. Run another Phase A "
        "experiment with a different seed."
    )

if phase_a_flip_rate < 0.30:
    print(
        "Warning: the best Phase A flip rate is "
        "below 30%. Another Phase A seed may provide "
        "a stronger starting checkpoint."
    )

NameError: name 'load_ppo_model' is not defined

In [30]:
phase_b_hub_checkpoint_callback = (
    HubCheckpointCallback(
        repo_id=(
            "KaptainKris92/"
            "ppo-LunarLander-v3-flip"
        ),
        every_timesteps=1_000_000,
    )
)

In [31]:
phase_b_run = train_ppo_experiment(
    experiment_name=(
        "ppo_lunarlander_flip_phase_b"
    ),
    total_timesteps=10_000_000,

    env_factory=(
        make_flip_landing_lander
    ),

    initial_model_path=(
        selected_phase_a_model_path
    ),

    n_envs=16,
    seed=42,

    # These document the loaded architecture.
    actor_layers=(128, 128),
    critic_layers=(128, 128),

    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.03,
    clip_range=0.2,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=20,

    # extra_callbacks=[
    #     phase_b_hub_checkpoint_callback
    # ],

    device="cpu",
)

NameError: name 'selected_phase_a_model_path' is not defined

### Evaluate model

In [ ]:
phase_b_checkpoint_paths = sorted(
    (
        phase_b_run["output_dir"]
        / "checkpoints"
    ).glob("*.zip"),
    key=checkpoint_step,
)

phase_b_candidate_paths = [
    Path(
        phase_b_run["best_model_path"]
    ),
    Path(
        phase_b_run["final_model_path"]
    ),
    *phase_b_checkpoint_paths,
]

phase_b_candidate_paths = list(
    dict.fromkeys(
        path
        for path in phase_b_candidate_paths
        if path.exists()
    )
)

phase_b_rows = []

for candidate_path in (
    phase_b_candidate_paths
):
    candidate_evaluation = (
        evaluate_flip_model(
            candidate_path,
            reward_config=(
                flip_landing_config
            ),
            n_episodes=20,
            starting_seed=20_000,
        )
    )

    phase_b_rows.append(
        {
            "model_path": str(
                candidate_path
            ),
            **candidate_evaluation[
                "summary"
            ],
        }
    )

phase_b_results = pd.DataFrame(
    phase_b_rows
)

phase_b_results = (
    phase_b_results
    .sort_values(
        [
            "flip_landing_rate",
            "landing_given_flip_rate",
            "recovery_given_flip_rate",
            "flip_completion_rate",
            "mean_original_reward",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    phase_b_results[
        [
            "model_path",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "flip_landing_rate",
            "landing_given_flip_rate",
            "mean_original_reward",
        ]
    ]
)

selected_model_path = Path(
    phase_b_results.loc[
        0,
        "model_path",
    ]
)

print(
    "Selected Phase B model:",
    selected_model_path,
)

flip_evaluation = evaluate_flip_model(
    selected_model_path,
    reward_config=flip_landing_config,
    n_episodes=100,
    starting_seed=20_000,
)

display(
    pd.Series(
        flip_evaluation["summary"],
        name="Final flip-and-land model",
    )
)

display(
    flip_evaluation["episodes"]
    .sort_values(
        [
            "flip_and_landed",
            "recovery_completed",
            "flip_completed",
            "original_reward",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .head(20)
)

### Video of best episode

In [ ]:
episode_results = (
    flip_evaluation["episodes"]
)

successful_flip_landings = (
    episode_results[
        episode_results[
            "flip_and_landed"
        ]
    ]
    .copy()
)

if successful_flip_landings.empty:
    video_seed = int(
        episode_results
        .sort_values(
            "shaped_reward",
            ascending=False,
        )
        .iloc[0]["seed"]
    )

    print(
        "No successful flip-and-land episode "
        "was found in the evaluation sample."
    )
    print(
        "Using the highest shaped-reward "
        "episode instead."
    )

else:
    video_seed = int(
        successful_flip_landings
        .sort_values(
            "original_reward",
            ascending=False,
        )
        .iloc[0]["seed"]
    )

    print(
        "Selected the successful flip-and-land "
        "episode with the highest original reward."
    )

print("Selected video seed:", video_seed)

In [ ]:
from pathlib import Path
import shutil

import gymnasium as gym


def record_flip_replay(
    model_or_path,
    *,
    output_path: str | Path,
    seed: int,
    reward_config: dict,
    deterministic: bool = True,
):
    """
    Record one complete episode from the custom flip environment.
    """

    model = load_ppo_model(
        model_or_path
    )

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_directory = (
        output_path.parent
        / "_flip_video_recording"
    )

    if temporary_directory.exists():
        shutil.rmtree(
            temporary_directory
        )

    temporary_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    video_env = make_flip_lander(
        render_mode="rgb_array",
        reward_config=reward_config,
    )

    video_env = gym.wrappers.RecordVideo(
        video_env,
        video_folder=str(
            temporary_directory
        ),
        episode_trigger=(
            lambda episode_number:
            episode_number == 0
        ),
        video_length=0,
        name_prefix=(
            "ppo-lunarlander-flip"
        ),
        disable_logger=True,
    )

    shaped_reward_total = 0.0
    original_reward_total = 0.0
    episode_steps = 0

    final_info = {}

    try:
        observation, info = video_env.reset(
            seed=int(seed)
        )

        terminated = False
        truncated = False

        while not (
            terminated or truncated
        ):
            action, _ = model.predict(
                observation,
                deterministic=deterministic,
            )

            action = int(
                np.asarray(
                    action
                ).item()
            )

            (
                observation,
                shaped_reward,
                terminated,
                truncated,
                info,
            ) = video_env.step(action)

            shaped_reward_total += float(
                shaped_reward
            )

            original_reward_total += float(
                info.get(
                    "original_reward",
                    shaped_reward,
                )
            )

            episode_steps += 1
            final_info = dict(info)

    finally:
        video_env.close()

    generated_videos = list(
        temporary_directory.glob(
            "*.mp4"
        )
    )

    if not generated_videos:
        raise FileNotFoundError(
            "Gymnasium did not generate "
            "an MP4 replay."
        )

    generated_video = max(
        generated_videos,
        key=lambda path:
        path.stat().st_mtime,
    )

    shutil.copy2(
        generated_video,
        output_path,
    )

    shutil.rmtree(
        temporary_directory,
        ignore_errors=True,
    )

    replay_details = {
        "path": str(output_path),
        "seed": int(seed),
        "shaped_reward": float(
            shaped_reward_total
        ),
        "original_reward": float(
            original_reward_total
        ),
        "steps": int(
            episode_steps
        ),
        "flip_completed": bool(
            final_info.get(
                "flip_completed",
                False,
            )
        ),
        "landed_safely": bool(
            final_info.get(
                "landed_safely",
                False,
            )
        ),
        "rotations_completed": float(
            final_info.get(
                "rotations_completed",
                0.0,
            )
        ),
        "recovery_completed": bool(
            final_info.get(
                "recovery_completed",
                False,
            )
        ),
        "custom_outcome": str(
            final_info.get(
                "custom_outcome",
                "unknown",
            )
        ),
        "flip_and_landed": (
            final_info.get(
                "custom_outcome"
            )
            == "flip_and_safe_landing"
        ),
    }

    print("Replay saved:", output_path)
    print(
        "Shaped reward:",
        f"{shaped_reward_total:.2f}",
    )
    print(
        "Original reward:",
        f"{original_reward_total:.2f}",
    )
    print(
        "Completed a flip:",
        replay_details[
            "flip_completed"
        ],
    )
    print(
        "Landed safely:",
        replay_details[
            "landed_safely"
        ],
    )
    print(
        "Flipped and landed:",
        replay_details[
            "flip_and_landed"
        ],
    )
    print(
        "Rotations completed:",
        f"{replay_details['rotations_completed']:.2f}",
    )

    return replay_details

In [ ]:
flip_replay = record_flip_replay(
    selected_model_path,
    output_path=(
        "/content/"
        "ppo-LunarLander-v3-"
        "flip-replay.mp4"
    ),
    seed=video_seed,
    reward_config=flip_landing_config,
)

flip_replay

In [ ]:
from IPython.display import (
    Video,
    display,
)


display(
    Video(
        flip_replay["path"],
        embed=True,
    )
)

### Upload as new model to Hugging Face

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import textwrap

from huggingface_hub import HfApi


DEFAULT_CHANGE_NOTES = [
    (
        "Reduced the combined rotation-progress and flip-completion "
        "reward so a flip followed by a crash is less attractive."
    ),
    (
        "Increased the pre-flip weight on the original LunarLander "
        "reward so the agent is less free to drift far from the pad "
        "while rotating."
    ),
    (
        "Added a one-off recovery reward for becoming upright and "
        "arresting angular velocity after the flip."
    ),
    (
        "Added dense post-flip potential-difference shaping for "
        "horizontal position, horizontal speed, attitude, angular "
        "speed, and leg contact."
    ),
    (
        "Added a failed-landing penalty and increased the terminal "
        "flip-and-safe-landing bonus."
    ),
    (
        "Added a recovery-completed observation flag. The wrapped "
        "observation now contains 11 values rather than the base "
        "environment's 8 values."
    ),
]


def _markdown_value(value) -> str:
    if isinstance(value, bool):
        return "true" if value else "false"

    if isinstance(value, float):
        return f"{value:g}"

    return str(value)


def _markdown_table(mapping: dict) -> str:
    rows = [
        "| Parameter | Value |",
        "|---|---:|",
    ]

    for key, value in mapping.items():
        rows.append(
            f"| `{key}` | {_markdown_value(value)} |"
        )

    return "\n".join(rows)


def upload_flip_model_to_hub(
    model_or_path,
    *,
    repo_id: str,
    flip_evaluation: dict,
    replay_details: dict,
    training_config_path: str | Path,
    reward_config: dict,
    reward_wrapper_path: str | Path,
    model_filename: str = (
        "ppo-LunarLander-v3-flip-128x128.zip"
    ),
    reward_version: str = "v2-post-flip-recovery",
    change_notes: list[str] | None = None,
    commit_message: str = (
        "Update PPO agent with post-flip recovery shaping"
    ),
    private: bool = False,
):
    """
    Upload a trained PPO checkpoint and the exact custom environment
    definition used to train and evaluate it.

    Reusing the same repo_id and filenames updates the current Hub
    files in a new commit; previous revisions remain in repository
    history.
    """

    if not reward_config:
        raise ValueError(
            "reward_config must contain the exact configuration "
            "used for training."
        )

    model = load_ppo_model(model_or_path)

    summary = dict(
        flip_evaluation["summary"]
    )
    episode_results = (
        flip_evaluation["episodes"].copy()
    )

    replay_path = Path(
        replay_details["path"]
    )
    training_config_path = Path(
        training_config_path
    )
    reward_wrapper_path = Path(
        reward_wrapper_path
    )

    required_paths = {
        "Replay": replay_path,
        "Training configuration": training_config_path,
        "Reward wrapper": reward_wrapper_path,
    }

    for label, path in required_paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f"{label} not found: {path}"
            )

    training_config = json.loads(
        training_config_path.read_text(
            encoding="utf-8"
        )
    )

    staging_directory = (
        Path("/content/hf-flip-model")
        / repo_id.replace("/", "__")
    )

    if staging_directory.exists():
        shutil.rmtree(staging_directory)

    staging_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    saved_model_path = (
        staging_directory / model_filename
    )
    model.save(str(saved_model_path))

    if not saved_model_path.exists():
        raise FileNotFoundError(
            "The PPO model was not saved at "
            f"{saved_model_path}"
        )

    shutil.copy2(
        replay_path,
        staging_directory / "replay.mp4",
    )
    shutil.copy2(
        training_config_path,
        staging_directory / "training_config.json",
    )
    shutil.copy2(
        reward_wrapper_path,
        staging_directory
        / "flip_landing_reward_wrapper.py",
    )

    (
        staging_directory / "reward_config.json"
    ).write_text(
        json.dumps(
            reward_config,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    episode_results.to_csv(
        staging_directory / "episode_results.csv",
        index=False,
    )

    notes = list(
        change_notes
        if change_notes is not None
        else DEFAULT_CHANGE_NOTES
    )

    results_payload = {
        "environment": {
            "base_environment": "LunarLander-v3",
            "custom_objective": (
                "Complete a full directed rotation, recover to "
                "a stable upright attitude, re-centre, and land "
                "safely."
            ),
            "reward_version": reward_version,
            "observation_space": {
                "base_dimensions": 8,
                "added_dimensions": [
                    "signed_rotation_progress",
                    "flip_completed",
                    "recovery_completed",
                ],
                "wrapped_dimensions": 11,
            },
            "reward_config": reward_config,
            "change_notes": notes,
        },
        "evaluation": summary,
        "evaluation_seeds": (
            episode_results["seed"]
            .astype(int)
            .tolist()
        ),
        "replay": {
            key: value
            for key, value in replay_details.items()
            if key != "path"
        },
    }

    (
        staging_directory / "results.json"
    ).write_text(
        json.dumps(
            results_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    config_payload = {
        "algorithm": "PPO",
        "library": "stable-baselines3",
        "base_environment": "LunarLander-v3",
        "model_filename": model_filename,
        "reward_version": reward_version,
        "reward_config_filename": "reward_config.json",
        "reward_wrapper_filename": (
            "flip_landing_reward_wrapper.py"
        ),
        "observation_dimensions": 11,
        "architecture": {
            "actor_layers": training_config.get(
                "actor_layers"
            ),
            "critic_layers": training_config.get(
                "critic_layers"
            ),
        },
        "evaluation": {
            "mean_shaped_reward": summary.get(
                "mean_shaped_reward"
            ),
            "mean_original_reward": summary.get(
                "mean_original_reward"
            ),
            "flip_completion_rate": summary.get(
                "flip_completion_rate"
            ),
            "recovery_completion_rate": summary.get(
                "recovery_completion_rate"
            ),
            "safe_landing_rate": summary.get(
                "safe_landing_rate"
            ),
            "flip_landing_rate": summary.get(
                "flip_landing_rate"
            ),
        },
    }

    (
        staging_directory / "config.json"
    ).write_text(
        json.dumps(
            config_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    actor_layers = training_config.get(
        "actor_layers",
        "Not supplied",
    )
    critic_layers = training_config.get(
        "critic_layers",
        "Not supplied",
    )

    video_url = (
        f"https://huggingface.co/{repo_id}"
        "/resolve/main/replay.mp4"
    )

    change_notes_markdown = "\n".join(
        f"{index}. {note}"
        for index, note in enumerate(
            notes,
            start=1,
        )
    )

    reward_table = _markdown_table(
        reward_config
    )

    optional_recovery_row = ""

    if (
        summary.get("recovery_completion_rate")
        is not None
    ):
        optional_recovery_row = (
            "| Post-flip recovery rate | "
            f"{summary['recovery_completion_rate']:.1%} |\n"
        )

    readme = textwrap.dedent(
        f"""
        ---
        library_name: stable-baselines3
        pipeline_tag: reinforcement-learning
        tags:
        - stable-baselines3
        - reinforcement-learning
        - deep-reinforcement-learning
        - PPO
        - LunarLander-v3
        - custom-reward
        - reward-shaping
        - actor-critic
        ---

        # PPO LunarLander flip, recover and land agent

        This repository contains a Stable-Baselines3 PPO
        actor-critic agent trained on a customised
        `LunarLander-v3` environment.

        The objective has three stages:

        1. complete one full rotation in a fixed direction;
        2. return upright and arrest the spin;
        3. re-centre, stabilise and land with both legs.

        Reward configuration version: `{reward_version}`.

        ## Changes in this upload

        {change_notes_markdown}

        ## Reward design

        Rotation progress is rewarded only when the agent reaches
        previously unseen progress. After the full rotation, a
        potential-difference reward encourages improvement in:

        - horizontal distance from the pad centre;
        - horizontal speed;
        - orientation;
        - angular speed;
        - leg contact.

        The terminal success bonus is awarded only after a completed
        flip and a safe landing. A completed flip followed by an
        unsuccessful landing receives the configured failed-landing
        penalty.

        {reward_table}

        ## Observation space

        The base `LunarLander-v3` observation contains 8 values.
        This wrapper appends:

        1. signed rotation progress;
        2. flip-completed flag;
        3. recovery-completed flag.

        The policy therefore expects **11 observation values** and
        cannot be used directly with an unwrapped
        `LunarLander-v3` environment.

        ## Evaluation

        Deterministic evaluation over
        {summary['n_episodes']} fixed-seed episodes:

        | Metric | Value |
        |---|---:|
        | Mean shaped reward | {summary['mean_shaped_reward']:.2f} |
        | Shaped reward standard deviation | {summary['std_shaped_reward']:.2f} |
        | Shaped course-style score | {summary['shaped_course_score']:.2f} |
        | Minimum shaped reward | {summary['min_shaped_reward']:.2f} |
        | Maximum shaped reward | {summary['max_shaped_reward']:.2f} |
        | Mean original reward | {summary['mean_original_reward']:.2f} |
        | Original reward standard deviation | {summary['std_original_reward']:.2f} |
        | Original course-style score | {summary['original_course_score']:.2f} |
        | Original reward >= 200 | {summary['original_reward_200_rate']:.1%} |
        | Full-rotation rate | {summary['flip_completion_rate']:.1%} |
        {optional_recovery_row}| Safe-landing rate | {summary['safe_landing_rate']:.1%} |
        | Flip-and-land rate | {summary['flip_landing_rate']:.1%} |
        | Minimum original reward | {summary['min_original_reward']:.2f} |
        | Maximum original reward | {summary['max_original_reward']:.2f} |

        Shaped rewards are not directly comparable with scores from
        the standard LunarLander environment.

        ## Architecture

        - Algorithm: PPO
        - Policy: MLP actor-critic
        - Actor hidden layers: `{actor_layers}`
        - Critic hidden layers: `{critic_layers}`

        ## Training configuration

        | Parameter | Value |
        |---|---:|
        | Training timesteps | {training_config.get('total_timesteps')} |
        | Parallel environments | {training_config.get('n_envs')} |
        | Learning rate | {training_config.get('learning_rate')} |
        | Rollout steps per environment | {training_config.get('n_steps')} |
        | Batch size | {training_config.get('batch_size')} |
        | Optimisation epochs | {training_config.get('n_epochs')} |
        | Gamma | {training_config.get('gamma')} |
        | GAE lambda | {training_config.get('gae_lambda')} |
        | Entropy coefficient | {training_config.get('ent_coef')} |
        | PPO clip range | {training_config.get('clip_range')} |
        | Training seed | {training_config.get('seed')} |

        ## Replay

        - Seed: `{replay_details['seed']}`
        - Original reward:
          `{replay_details['original_reward']:.2f}`
        - Shaped reward:
          `{replay_details['shaped_reward']:.2f}`
        - Rotations completed:
          `{replay_details['rotations_completed']:.2f}`
        - Flip completed:
          `{replay_details['flip_completed']}`
        - Recovery completed:
          `{replay_details.get('recovery_completed', 'Not recorded')}`
        - Landed safely:
          `{replay_details['landed_safely']}`

        <video controls autoplay loop muted width="640">
          <source src="{video_url}" type="video/mp4">
        </video>

        ## Load the model

        The wrapper and reward configuration are included in this
        repository because the policy requires the augmented
        11-value observation.

        ```python
        import json
        import sys
        from pathlib import Path

        from huggingface_hub import hf_hub_download
        from stable_baselines3 import PPO

        checkpoint = hf_hub_download(
            repo_id="{repo_id}",
            filename="{model_filename}",
        )
        wrapper_file = hf_hub_download(
            repo_id="{repo_id}",
            filename="flip_landing_reward_wrapper.py",
        )
        reward_file = hf_hub_download(
            repo_id="{repo_id}",
            filename="reward_config.json",
        )

        sys.path.insert(
            0,
            str(Path(wrapper_file).parent),
        )

        from flip_landing_reward_wrapper import (
            make_flip_lander,
        )

        reward_config = json.loads(
            Path(reward_file).read_text(
                encoding="utf-8"
            )
        )

        env = make_flip_lander(
            reward_config=reward_config
        )

        model = PPO.load(
            checkpoint,
            env=env,
            device="cpu",
        )
        ```
        """
    ).strip()

    (
        staging_directory / "README.md"
    ).write_text(
        readme,
        encoding="utf-8",
    )

    print("Prepared files:")

    for path in sorted(
        staging_directory.iterdir()
    ):
        print(
            "-",
            path.name,
            f"({path.stat().st_size / 1024:.1f} KiB)",
        )

    api = HfApi()

    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=private,
        exist_ok=True,
    )

    upload_result = api.upload_folder(
        folder_path=str(staging_directory),
        repo_id=repo_id,
        repo_type="model",
        commit_message=commit_message,
    )

    print()
    print("Upload complete:", upload_result)
    print(
        "Model repository:",
        f"https://huggingface.co/{repo_id}",
    )

    return {
        "repo_id": repo_id,
        "staging_directory": staging_directory,
        "summary": summary,
        "reward_version": reward_version,
        "upload_result": upload_result,
    }


In [ ]:
upload_result = upload_flip_model_to_hub(
    selected_model_path,

    repo_id=(
        "KaptainKris92/"
        "ppo-LunarLander-v3-flip"
    ),

    flip_evaluation=flip_evaluation,
    replay_details=flip_replay,

    training_config_path=(
        phase_b_run["config_path"]
    ),

    reward_config=flip_landing_config,

    reward_wrapper_path=(
        REWARD_WRAPPER_PATH
    ),

    model_filename=(
        "ppo-LunarLander-v3-"
        "flip.zip"
    ),

    change_notes=[
        (
            "Introduced a two-stage curriculum: "
            "Phase A acquires the full rotation and "
            "Phase B continues from the strongest "
            "flip-capable checkpoint."
        ),
        (
            "Penalised safe landing without first "
            "completing the required rotation."
        ),
        (
            "Added vertical-speed control to the "
            "post-flip recovery potential."
        ),
        (
            "Multiplied the original LunarLander "
            "reward by 3 after flip completion."
        ),
        (
            "Selected the final checkpoint using "
            "flip-and-land and recovery metrics "
            "rather than mean shaped reward alone."
        ),
    ],

    commit_message=(
        "Train PPO with flip-acquisition "
        "and post-flip landing curriculum"
    ),
)